# OptiCrop — Model Training & EDA

This notebook explores the crop recommendation dataset and trains the ML models used by OptiCrop.

**Dataset:** N, P, K, temperature, humidity, pH, rainfall → crop label (22 classes)

In [ ]:
import os
import sys
import urllib.request

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

from app.ml.preprocessor import CropPreprocessor
from app.ml.algorithms.knn import KNNCropModel
from app.ml.algorithms.logistic import LogisticCropModel
from app.ml.algorithms.decision_tree import DecisionTreeCropModel
from app.ml.algorithms.random_forest import RandomForestCropModel
from app.ml.algorithms.kmeans import KMeansCropModel

DATA_PATH = '../data/Crop_recommendation.csv'
DATA_URL = (
    'https://raw.githubusercontent.com/Gladiator07/Harvestify/master/'
    'Data-processed/crop_recommendation.csv'
)

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Dataset

In [ ]:
os.makedirs('../data', exist_ok=True)
if not os.path.exists(DATA_PATH):
    print('Downloading dataset...')
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

preprocessor = CropPreprocessor()
df = preprocessor.load_data(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Crop distribution:')
print(df['label'].value_counts())

df.describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['label'].value_counts().plot(kind='bar', ax=axes[0, 0], color='#74c69d')
axes[0, 0].set_title('Crop Class Distribution')
axes[0, 0].tick_params(axis='x', rotation=45)

sns.histplot(df['N'], kde=True, ax=axes[0, 1], color='#1a472a')
axes[0, 1].set_title('Nitrogen Distribution')

sns.scatterplot(data=df, x='temperature', y='humidity', hue='label', alpha=0.5, ax=axes[1, 0])
axes[1, 0].set_title('Temperature vs Humidity by Crop')

sns.boxplot(data=df, x='label', y='rainfall', ax=axes[1, 1])
axes[1, 1].set_title('Rainfall by Crop')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
corr = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']].corr()
sns.heatmap(corr, annot=True, cmap='Greens', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

## 3. Preprocess & Split

In [ ]:
X, y = preprocessor.prepare_features(df, fit=True)
X_train, X_test, y_train, y_test = preprocessor.split_data(X, y)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')
print(f'Features: {X.shape[1]}')
print(f'Classes: {len(preprocessor.label_encoder.classes_)}')

## 4. Train All Models

In [ ]:
os.makedirs('../models', exist_ok=True)
preprocessor.save('../models/scaler.pkl')

models = {
    'KNN': KNNCropModel(n_neighbors=5),
    'Logistic Regression': LogisticCropModel(),
    'Decision Tree': DecisionTreeCropModel(),
    'Random Forest': RandomForestCropModel(),
    'KMeans': KMeansCropModel(),
}

results = {}
for name, model in models.items():
    print(f'Training {name}...')
    model.train(X_train, y_train)
    accuracy = model.evaluate(X_test, y_test)
    results[name] = accuracy
    model.save()
    print(f'  Accuracy: {accuracy:.4f}')

## 5. Model Comparison

In [ ]:
results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Accuracy'])
results_df = results_df.sort_values('Accuracy', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(results_df['Model'], results_df['Accuracy'], color=['#74c69d', '#40916c', '#2d6a4f', '#1a472a', '#f4a261'])
ax.set_ylabel('Accuracy')
ax.set_title('OptiCrop Model Accuracy Comparison')
ax.set_ylim(0, 1.05)
for bar, acc in zip(bars, results_df['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{acc:.2%}', ha='center', fontsize=10)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

results_df

## 6. Sample Prediction

In [ ]:
from app.ml.predictor import CropPredictor

predictor = CropPredictor()
sample = predictor.predict(90, 42, 43, 25, 80, 6.5, 200, 'random_forest')
print(f"Predicted crop: {sample.get('emoji', '')} {sample.get('crop', 'unknown')}")
print(f"Confidence: {sample.get('confidence', 'N/A')}")
print(f"Description: {sample.get('description', '')}")